In [ ]:
!pip install decord -q

# Standard library
import os
import pickle

# Third-party
import numpy as np
from tqdm import tqdm
import torch
import torchvision.transforms.functional as TF

# Video processing
import decord
from decord import VideoReader, cpu

# Optimize decord + torch performance
decord.bridge.set_bridge("torch")
torch.backends.cudnn.benchmark = True

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

print("Environment ready.")


In [ ]:
# ---------------------------------------------------------
# CELL: Safe Video -> Tensor Preprocessing (1 video = 1 tensor)
# ---------------------------------------------------------
from pathlib import Path

VALID_VIDEO_EXTS = {".mp4", ".avi", ".mkv", ".webm", ".mov", ".MP4", ".AVI", ".MKV", ".WEBM", ".MOV"}

def list_real_video_files(video_root):
    """Return only real video files; ignore hidden/temp/system files."""
    root = Path(video_root)
    if not root.exists():
        raise FileNotFoundError(f"Video root not found: {video_root}")

    files = []
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        if p.name.startswith("."):
            continue
        if p.suffix not in VALID_VIDEO_EXTS:
            continue
        files.append(p)

    files = sorted(files, key=lambda x: str(x))
    return files

def load_annotations(annotation_pkl_path):
    """Load Chalearn annotation file into {video_id: [5 traits]}."""
    traits = ['extraversion', 'agreeableness', 'conscientiousness', 'neuroticism', 'openness']
    with open(annotation_pkl_path, "rb") as f:
        raw = pickle.load(f, encoding="latin1")

    out = {}
    for vid_key in raw[traits[0]].keys():
        vid = Path(str(vid_key)).stem
        out[vid] = [float(raw[t][vid_key]) for t in traits]
    return out

def sample_indices_uniform(num_frames, max_frames):
    """Uniform frame sampling within one video. Never creates extra samples."""
    if num_frames <= 0:
        return []
    if num_frames >= max_frames:
        return np.linspace(0, num_frames - 1, max_frames, dtype=int).tolist()

    # Pad by repeating last frame index only inside the same sample.
    idx = list(range(num_frames))
    idx.extend([num_frames - 1] * (max_frames - num_frames))
    return idx

def decode_video_as_single_tensor(video_path, max_frames=8, resize=140):
    """Decode one full video into exactly one tensor sample."""
    vr = VideoReader(str(video_path), ctx=cpu(0))
    n = len(vr)
    if n == 0:
        raise RuntimeError(f"Empty video: {video_path}")

    idx = sample_indices_uniform(n, max_frames=max_frames)
    frames = vr.get_batch(idx).permute(0, 3, 1, 2).float() / 255.0
    frames = TF.resize(frames, resize, antialias=True)
    frames = TF.center_crop(frames, [resize, resize])
    return frames  # [T, C, H, W]

def preprocess_split_one_tensor_per_video(
    split_name,
    video_root,
    annotation_pkl_path,
    output_dir,
    max_frames=8,
    resize=140,
    overwrite=False,
    strict_annotation_match=True,
):
    """
    Enforces one-output-per-video policy for a single split.
    - No clip extraction windows
    - No DataLoader workers (avoids multiprocessing duplicate saves)
    - Stable output name: <video_id>.pt
    """
    os.makedirs(output_dir, exist_ok=True)

    videos = list_real_video_files(video_root)
    annotations = load_annotations(annotation_pkl_path)

    seen_ids = set()
    saved = 0
    skipped_no_label = 0
    skipped_duplicate_input_name = 0
    failed = 0

    print(f"[{split_name}] videos discovered: {len(videos)}")
    print(f"[{split_name}] annotation entries: {len(annotations)}")

    for vp in tqdm(videos, desc=f"Preprocess {split_name}"):
        vid = vp.stem

        # Detect duplicate input names immediately (same stem appears twice).
        if vid in seen_ids:
            skipped_duplicate_input_name += 1
            continue
        seen_ids.add(vid)

        if strict_annotation_match and vid not in annotations:
            skipped_no_label += 1
            continue

        out_path = Path(output_dir) / f"{vid}.pt"
        if out_path.exists() and not overwrite:
            continue

        try:
            frames = decode_video_as_single_tensor(vp, max_frames=max_frames, resize=resize)
            labels = annotations.get(vid, [0.0] * 5)
            torch.save({
                "video_id": vid,
                "frames": frames.cpu(),
                "labels": torch.tensor(labels, dtype=torch.float32),
            }, out_path)
            saved += 1
        except Exception as e:
            failed += 1
            # Keep processing; print compact error.
            print(f"Failed: {vp.name} | {str(e)[:160]}")

    output_files = sorted(Path(output_dir).glob("*.pt"))
    output_ids = [p.stem for p in output_files]
    duplicate_output_ids = len(output_ids) - len(set(output_ids))

    print("\n--- Split Summary ---")
    print(f"Split: {split_name}")
    print(f"Original discovered videos: {len(videos)}")
    print(f"Unique input video ids: {len(seen_ids)}")
    print(f"Saved tensor files: {len(output_files)}")
    print(f"Saved in this run: {saved}")
    print(f"Skipped (no annotation): {skipped_no_label}")
    print(f"Skipped (duplicate input name): {skipped_duplicate_input_name}")
    print(f"Failed decode: {failed}")
    print(f"Duplicate output ids: {duplicate_output_ids}")

    # Safety checks requested by user.
    if len(output_files) > len(seen_ids):
        raise RuntimeError(
            f"Output tensors ({len(output_files)}) exceed unique videos ({len(seen_ids)})."
        )
    if duplicate_output_ids != 0:
        raise RuntimeError("Duplicate output tensor names detected.")

    return {
        "split": split_name,
        "video_count": len(videos),
        "unique_video_ids": len(seen_ids),
        "saved_files": len(output_files),
        "saved_now": saved,
        "skipped_no_label": skipped_no_label,
        "skipped_duplicate_input_name": skipped_duplicate_input_name,
        "failed": failed,
    }

In [ ]:
# ------------------- Configure your split-specific paths -------------------
# IMPORTANT: Train and validation are already separate folders; we do NOT split train.
TRAIN_VIDEO_ROOT = "/kaggle/input/first-impressions/train"
VAL_VIDEO_ROOT = "/kaggle/input/first-impressions/validation"

TRAIN_ANNOTATION_PKL = "/kaggle/input/annotation-files/annotation-training.pkl"
VAL_ANNOTATION_PKL = "/kaggle/input/annotation-files/annotation-validation.pkl"

TRAIN_TENSOR_OUT = "/kaggle/working/preprocessed/train_tensors"
VAL_TENSOR_OUT = "/kaggle/working/preprocessed/val_tensors"

# Run preprocessing (one tensor per original video).
train_summary = preprocess_split_one_tensor_per_video(
    split_name="train",
    video_root=TRAIN_VIDEO_ROOT,
    annotation_pkl_path=TRAIN_ANNOTATION_PKL,
    output_dir=TRAIN_TENSOR_OUT,
    max_frames=8,
    resize=140,
    overwrite=False,
    strict_annotation_match=True,
)

val_summary = preprocess_split_one_tensor_per_video(
    split_name="validation",
    video_root=VAL_VIDEO_ROOT,
    annotation_pkl_path=VAL_ANNOTATION_PKL,
    output_dir=VAL_TENSOR_OUT,
    max_frames=8,
    resize=140,
    overwrite=False,
    strict_annotation_match=True,
)

print("\nTrain summary:", train_summary)
print("Validation summary:", val_summary)